# Benin — EDA Notebook

In [ ]:
from pathlib import Path
import os

# auto-detect repo root (folder containing "data" folder)
p = Path.cwd()
while not (p / "data").exists():
    p = p.parent
os.chdir(p)

print("Working directory set to:", os.getcwd())


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({"figure.figsize": (12, 5)})

RAW_PATH = "data/benin.csv"
CLEAN_PATH = "data/benin_clean.csv"

df = pd.read_csv(RAW_PATH, parse_dates=["Timestamp"])
df.sort_values("Timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)

df.head()


## 1. Summary Statistics

In [ ]:

display(df.describe(include='all'))

na_counts = df.isna().sum()
na_pct = (na_counts / len(df) * 100).round(2)
display(pd.DataFrame({"Missing": na_counts, "Percent": na_pct}))


## 2. Outlier Detection using Z-scores

In [ ]:

numeric_cols = ["GHI","DNI","DHI","ModA","ModB","WS","WSgust"]

for col in numeric_cols:
    if col in df.columns:
        df[col+"_z"] = (df[col] - df[col].mean()) / df[col].std()

outlier_summary = {col: df[df[col+"_z"].abs() > 3].shape[0] for col in numeric_cols if col+"_z" in df.columns}
outlier_summary


## 3. Data Cleaning (Median Imputation)

In [ ]:

key_cols = ["GHI","DNI","DHI","ModA","ModB","Tamb","RH","WS","WSgust","BP","TModA","TModB"]

for col in key_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

df.to_csv(CLEAN_PATH, index=False)
CLEAN_PATH


## 4. Time Series Plots

In [ ]:

for col in ["GHI", "DNI", "DHI", "Tamb"]:
    if col in df.columns:
        plt.figure()
        plt.plot(df["Timestamp"], df[col])
        plt.title(f"{col} over time")
        plt.xlabel("Timestamp")
        plt.ylabel(col)
        plt.show()


## 5. Cleaning Impact on ModA & ModB

In [ ]:

if "Cleaning" in df.columns:
    grp = df.groupby("Cleaning")[["ModA","ModB"]].mean()
    display(grp)
    grp.plot(kind="bar")
    plt.title("Cleaning impact")
    plt.show()


## 6. Correlation Heatmap

In [ ]:

numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12,8))
sns.heatmap(numeric_df.corr(), annot=False, cmap="viridis")
plt.title("Correlation Heatmap")
plt.show()


## 7. Scatter Plots

In [ ]:

pairs = [("WS","GHI"), ("WSgust","GHI"), ("WD","GHI"), ("RH","Tamb"), ("RH","GHI")]

for x,y in pairs:
    if x in df.columns and y in df.columns:
        plt.figure()
        plt.scatter(df[x], df[y], alpha=0.3)
        plt.xlabel(x)
        plt.ylabel(y)
        plt.title(f"{x} vs {y}")
        plt.show()


## 8. Histograms

In [ ]:

for col in ["GHI","WS"]:
    if col in df.columns:
        plt.figure()
        plt.hist(df[col], bins=30)
        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.show()


## 9. Wind Rose Plot

In [ ]:

try:
    from windrose import WindroseAxes
    if "WS" in df.columns and "WD" in df.columns:
        ax = WindroseAxes.from_ax()
        ax.bar(df["WD"], df["WS"], normed=True, opening=0.8, edgecolor='white')
        ax.set_legend()
        plt.title("Wind Rose")
        plt.show()
except Exception as e:
    print("Windrose not available:", e)


## 10. Bubble Chart (GHI vs Tamb, bubble = RH)

In [ ]:

if "GHI" in df.columns and "Tamb" in df.columns and "RH" in df.columns:
    plt.figure()
    plt.scatter(df["GHI"], df["Tamb"], s=df["RH"]+1, alpha=0.3)
    plt.xlabel("GHI")
    plt.ylabel("Tamb")
    plt.title("Bubble Chart: GHI vs Tamb (bubble = RH)")
    plt.show()
